# MongoDB Change Streams to Snowflake

One notebook that is the whole pipeline: it reads MongoDB change streams with `pymongo`, lands every
event in Snowflake, keeps a current-state table per document, and exposes typed column views and a
lag view. Schedule this notebook and it runs top to bottom on each run.

Pair-programmed by SE Community + Cortex Code

| Requirement | Where it is handled |
| --- | --- |
| Map documents into columns | `COLUMN_MAP` rows generate one typed view per collection (section 6) |
| Know how far behind you are | `CHECKPOINTS.CAUGHT_UP_AT` and the `SYNC_LAG` view (section 7) |
| Resume without a full reload | `CHECKPOINTS.RESUME_TOKEN` is passed to `watch(resume_after=...)` (section 4) |

**Delivery semantics:** at least once. A run writes events, then saves the resume token. If a run dies
between the two, the next run replays those events and the MERGE ignores anything not newer than what
it already holds.

## 0. One-time setup (run in a SQL worksheet, not in this notebook)

Secrets and external access integrations are attached to the notebook service, so they must exist
before the notebook starts. Run this once with a role that has `CREATE INTEGRATION` (usually
ACCOUNTADMIN). Replace the host list and the connection string first.

```sql
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS SNOWFLAKE_EXAMPLE;
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC;

-- Every replica set member, host:port. List each host explicitly.
CREATE OR REPLACE NETWORK RULE SNOWFLAKE_EXAMPLE.MONGODB_CDC.MONGODB_EGRESS_RULE
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('<member-0-host>:27017', '<member-1-host>:27017', '<member-2-host>:27017');

-- User-defined PyPI rule. The Snowflake-managed PYPI_RULE is usable only by ACCOUNTADMIN,
-- which makes scheduled runs fail.
CREATE OR REPLACE NETWORK RULE SNOWFLAKE_EXAMPLE.MONGODB_CDC.PYPI_EGRESS_RULE
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('pypi.org:443', 'files.pythonhosted.org:443');

CREATE OR REPLACE SECRET SNOWFLAKE_EXAMPLE.MONGODB_CDC.MONGODB_URI
  TYPE = GENERIC_STRING
  SECRET_STRING = '<mongodb-connection-string>';

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION MONGODB_CDC_EAI
  ALLOWED_NETWORK_RULES = (SNOWFLAKE_EXAMPLE.MONGODB_CDC.MONGODB_EGRESS_RULE,
                           SNOWFLAKE_EXAMPLE.MONGODB_CDC.PYPI_EGRESS_RULE)
  ALLOWED_AUTHENTICATION_SECRETS = (SNOWFLAKE_EXAMPLE.MONGODB_CDC.MONGODB_URI)
  ENABLED = TRUE;

-- Scheduled runs use the owner's DEFAULT role. Grant that role what the notebook needs.
SET run_role = (SELECT CURRENT_ROLE());  -- change to the default role of the scheduling user
GRANT USAGE ON INTEGRATION MONGODB_CDC_EAI TO ROLE IDENTIFIER($run_role);
GRANT USAGE ON DATABASE SNOWFLAKE_EXAMPLE TO ROLE IDENTIFIER($run_role);
GRANT ALL ON SCHEMA SNOWFLAKE_EXAMPLE.MONGODB_CDC TO ROLE IDENTIFIER($run_role);
GRANT READ ON SECRET SNOWFLAKE_EXAMPLE.MONGODB_CDC.MONGODB_URI TO ROLE IDENTIFIER($run_role);
```

Then edit the notebook service: attach `MONGODB_CDC_EAI` and the `MONGODB_URI` secret.
MongoDB must be a replica set or sharded cluster; change streams do not exist on a standalone server.

## 1. Objects (idempotent, safe on every run)

In [ ]:
%%sql -r setup_objects
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC;

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.COLLECTIONS (
    SOURCE_DB          STRING  NOT NULL,
    SOURCE_COLLECTION  STRING  NOT NULL,
    TARGET_VIEW        STRING  NOT NULL,
    ENABLED            BOOLEAN NOT NULL DEFAULT TRUE,
    CONSTRAINT PK_COLLECTIONS PRIMARY KEY (SOURCE_DB, SOURCE_COLLECTION)
);

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.CHECKPOINTS (
    SOURCE_DB              STRING NOT NULL,
    SOURCE_COLLECTION      STRING NOT NULL,
    RESUME_TOKEN           STRING,
    LAST_EVENT_CLUSTER_TS  NUMBER,
    LAST_EVENT_CLUSTER_INC NUMBER,
    CAUGHT_UP_AT           TIMESTAMP_TZ,
    SNAPSHOT_COMPLETED_AT  TIMESTAMP_TZ,
    STATUS                 STRING,
    STATUS_MESSAGE         STRING,
    UPDATED_AT             TIMESTAMP_TZ,
    CONSTRAINT PK_CHECKPOINTS PRIMARY KEY (SOURCE_DB, SOURCE_COLLECTION)
);

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.RAW_EVENTS (
    RUN_ID            STRING,
    EVENT_SEQ         NUMBER,
    SOURCE_DB         STRING,
    SOURCE_COLLECTION STRING,
    DOC_ID            STRING,
    OPERATION         STRING,
    CLUSTER_TS        NUMBER,
    CLUSTER_INC       NUMBER,
    DOCUMENT          VARIANT,
    LOADED_AT         TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.DOCUMENTS (
    SOURCE_DB         STRING,
    SOURCE_COLLECTION STRING,
    DOC_ID            STRING,
    DOCUMENT          VARIANT,
    IS_DELETED        BOOLEAN,
    CLUSTER_TS        NUMBER,
    CLUSTER_INC       NUMBER,
    UPDATED_AT        TIMESTAMP_TZ
);

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.COLUMN_MAP (
    SOURCE_DB         STRING NOT NULL,
    SOURCE_COLLECTION STRING NOT NULL,
    COLUMN_NAME       STRING NOT NULL,
    JSON_PATH         STRING NOT NULL,
    DATA_TYPE         STRING NOT NULL,
    ORDINAL           NUMBER DEFAULT 100
);

CREATE TABLE IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.RUN_LOG (
    RUN_ID            STRING,
    SOURCE_DB         STRING,
    SOURCE_COLLECTION STRING,
    MODE              STRING,
    EVENTS_WRITTEN    NUMBER,
    STATUS            STRING,
    MESSAGE           STRING,
    STARTED_AT        TIMESTAMP_TZ,
    FINISHED_AT       TIMESTAMP_TZ
);

CREATE STREAM IF NOT EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC.RAW_EVENTS_STREAM
    ON TABLE SNOWFLAKE_EXAMPLE.MONGODB_CDC.RAW_EVENTS
    APPEND_ONLY = TRUE;

## 2. Register collections and column mappings

Edit these two statements for your collections. `MERGE` keeps them idempotent, so the scheduled run
does not duplicate rows. `JSON_PATH` uses Snowflake `GET_PATH` syntax, for example `profile.email`
or `items[0].sku`.

In [ ]:
%%sql -r register_collections
MERGE INTO SNOWFLAKE_EXAMPLE.MONGODB_CDC.COLLECTIONS t
USING (
    SELECT column1 AS SOURCE_DB, column2 AS SOURCE_COLLECTION, column3 AS TARGET_VIEW
    FROM VALUES
        ('sample_mflix', 'users', 'MFLIX_USERS')
) s
ON t.SOURCE_DB = s.SOURCE_DB AND t.SOURCE_COLLECTION = s.SOURCE_COLLECTION
WHEN NOT MATCHED THEN INSERT (SOURCE_DB, SOURCE_COLLECTION, TARGET_VIEW, ENABLED)
    VALUES (s.SOURCE_DB, s.SOURCE_COLLECTION, s.TARGET_VIEW, TRUE);

In [ ]:
%%sql -r register_columns
MERGE INTO SNOWFLAKE_EXAMPLE.MONGODB_CDC.COLUMN_MAP t
USING (
    SELECT column1 AS SOURCE_DB, column2 AS SOURCE_COLLECTION, column3 AS COLUMN_NAME,
           column4 AS JSON_PATH, column5 AS DATA_TYPE, column6 AS ORDINAL
    FROM VALUES
        ('sample_mflix', 'users', 'NAME',  'name',  'STRING', 10),
        ('sample_mflix', 'users', 'EMAIL', 'email', 'STRING', 20)
) s
ON  t.SOURCE_DB = s.SOURCE_DB
AND t.SOURCE_COLLECTION = s.SOURCE_COLLECTION
AND t.COLUMN_NAME = s.COLUMN_NAME
WHEN MATCHED THEN UPDATE SET JSON_PATH = s.JSON_PATH, DATA_TYPE = s.DATA_TYPE, ORDINAL = s.ORDINAL
WHEN NOT MATCHED THEN INSERT (SOURCE_DB, SOURCE_COLLECTION, COLUMN_NAME, JSON_PATH, DATA_TYPE, ORDINAL)
    VALUES (s.SOURCE_DB, s.SOURCE_COLLECTION, s.COLUMN_NAME, s.JSON_PATH, s.DATA_TYPE, s.ORDINAL);

## 3. Driver

In [ ]:
# Needs MONGODB_CDC_EAI (with the PyPI rule) attached to the notebook service.
%pip install --quiet "pymongo>=4.6,<5"

## 4. Sync: snapshot once, then resume from the saved token

Settings below control one run. Keep `RUN_SECONDS` well under the schedule interval so runs never
overlap.

- **First run for a collection:** opens a change stream to capture a starting token, copies every
  document as `snapshot` rows, then saves the token. Changes made during the copy are replayed on the
  next run, and the MERGE keeps the newest version.
- **Every later run:** `watch(resume_after=token)` reads only what changed since the last run.
- **Quiet collections:** when the stream has nothing new, the run still saves the stream's latest
  token, so the saved position keeps moving forward with the oplog.
- **Token older than the oplog** (MongoDB error 286, `ChangeStreamHistoryLost`): with
  `AUTO_RESNAPSHOT = True` the collection is snapshotted again and documents missing from the new
  snapshot are marked deleted. With `False`, the checkpoint is marked `HISTORY_LOST` and left for a
  person.

In [ ]:
SCHEMA = "SNOWFLAKE_EXAMPLE.MONGODB_CDC"
SECRET_PATH = "snowflake_example/mongodb_cdc/mongodb_uri"  # normalized db/schema/secret name
RUN_SECONDS = 240           # wall-clock budget for the whole sync step
FLUSH_EVERY = 5000          # events (or snapshot docs) per write to Snowflake
AUTO_RESNAPSHOT = True

In [ ]:
import time
import uuid

from bson import ObjectId, json_util
from bson.json_util import RELAXED_JSON_OPTIONS
from pymongo import MongoClient
from pymongo.errors import OperationFailure, PyMongoError
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.secrets import get_generic_secret_string
from snowflake.snowpark.types import LongType, StringType, StructField, StructType

session = get_active_session()
RUN_ID = str(uuid.uuid4())
HISTORY_LOST = 286
CHANGE_OPS = {"insert", "update", "replace", "delete"}
STOP_OPS = {"drop", "rename", "dropDatabase", "invalidate"}

LANDING_SCHEMA = StructType([
    StructField("RUN_ID", StringType()),
    StructField("EVENT_SEQ", LongType()),
    StructField("SOURCE_DB", StringType()),
    StructField("SOURCE_COLLECTION", StringType()),
    StructField("DOC_ID", StringType()),
    StructField("OPERATION", StringType()),
    StructField("CLUSTER_TS", LongType()),
    StructField("CLUSTER_INC", LongType()),
    StructField("DOCUMENT_JSON", StringType()),
])


def doc_id_of(value):
    if isinstance(value, (ObjectId, str, int)):
        return str(value)
    return json_util.dumps(value, json_options=RELAXED_JSON_OPTIONS)


def to_json(doc):
    return None if doc is None else json_util.dumps(doc, json_options=RELAXED_JSON_OPTIONS)


class Writer:
    # Buffers rows and appends them to RAW_EVENTS through a temporary landing table.
    def __init__(self, db, coll):
        self.db, self.coll, self.rows, self.seq, self.total = db, coll, [], 0, 0

    def add(self, doc_id, op, ts, inc, document):
        self.seq += 1
        self.rows.append([RUN_ID, self.seq, self.db, self.coll, doc_id, op, ts, inc, to_json(document)])
        if len(self.rows) >= FLUSH_EVERY:
            self.flush()

    def flush(self):
        if not self.rows:
            return
        landing = f"{SCHEMA}.RAW_EVENTS_LANDING"
        session.create_dataframe(self.rows, schema=LANDING_SCHEMA).write.mode("overwrite") \
            .save_as_table(landing, table_type="temporary")
        session.sql(f"""
            INSERT INTO {SCHEMA}.RAW_EVENTS
                (RUN_ID, EVENT_SEQ, SOURCE_DB, SOURCE_COLLECTION, DOC_ID, OPERATION,
                 CLUSTER_TS, CLUSTER_INC, DOCUMENT)
            SELECT RUN_ID, EVENT_SEQ, SOURCE_DB, SOURCE_COLLECTION, DOC_ID, OPERATION,
                   CLUSTER_TS, CLUSTER_INC, PARSE_JSON(DOCUMENT_JSON)
            FROM {landing}
        """).collect()
        self.total += len(self.rows)
        self.rows = []


def save_checkpoint(db, coll, token, status, message=None, last_ts=None, last_inc=None,
                    caught_up=False, snapshot_done=False):
    session.sql(f"""
        MERGE INTO {SCHEMA}.CHECKPOINTS t
        USING (SELECT ? AS SOURCE_DB, ? AS SOURCE_COLLECTION, ? AS RESUME_TOKEN, ? AS STATUS,
                      ? AS STATUS_MESSAGE, ?::NUMBER AS LAST_TS, ?::NUMBER AS LAST_INC,
                      ?::BOOLEAN AS CAUGHT_UP, ?::BOOLEAN AS SNAPSHOT_DONE) s
        ON t.SOURCE_DB = s.SOURCE_DB AND t.SOURCE_COLLECTION = s.SOURCE_COLLECTION
        WHEN MATCHED THEN UPDATE SET
            RESUME_TOKEN           = s.RESUME_TOKEN,
            STATUS                 = s.STATUS,
            STATUS_MESSAGE         = s.STATUS_MESSAGE,
            LAST_EVENT_CLUSTER_TS  = COALESCE(s.LAST_TS, t.LAST_EVENT_CLUSTER_TS),
            LAST_EVENT_CLUSTER_INC = COALESCE(s.LAST_INC, t.LAST_EVENT_CLUSTER_INC),
            CAUGHT_UP_AT           = IFF(s.CAUGHT_UP, CURRENT_TIMESTAMP(), t.CAUGHT_UP_AT),
            SNAPSHOT_COMPLETED_AT  = IFF(s.SNAPSHOT_DONE, CURRENT_TIMESTAMP(), t.SNAPSHOT_COMPLETED_AT),
            UPDATED_AT             = CURRENT_TIMESTAMP()
        WHEN NOT MATCHED THEN INSERT
            (SOURCE_DB, SOURCE_COLLECTION, RESUME_TOKEN, STATUS, STATUS_MESSAGE,
             LAST_EVENT_CLUSTER_TS, LAST_EVENT_CLUSTER_INC, CAUGHT_UP_AT, SNAPSHOT_COMPLETED_AT, UPDATED_AT)
        VALUES
            (s.SOURCE_DB, s.SOURCE_COLLECTION, s.RESUME_TOKEN, s.STATUS, s.STATUS_MESSAGE,
             s.LAST_TS, s.LAST_INC, IFF(s.CAUGHT_UP, CURRENT_TIMESTAMP(), NULL),
             IFF(s.SNAPSHOT_DONE, CURRENT_TIMESTAMP(), NULL), CURRENT_TIMESTAMP())
    """, params=[db, coll, token, status, message, last_ts, last_inc, caught_up, snapshot_done]).collect()


def log_run(db, coll, mode, events, status, message, started):
    session.sql(f"""
        INSERT INTO {SCHEMA}.RUN_LOG
            (RUN_ID, SOURCE_DB, SOURCE_COLLECTION, MODE, EVENTS_WRITTEN, STATUS, MESSAGE, STARTED_AT, FINISHED_AT)
        SELECT ?, ?, ?, ?, ?, ?, ?, TO_TIMESTAMP_TZ(?::NUMBER), CURRENT_TIMESTAMP()
    """, params=[RUN_ID, db, coll, mode, events, status, message, int(started)]).collect()


def snapshot(client, db, coll, deadline):
    # Token first, so nothing between the token and the copy is lost.
    collection = client[db][coll]
    with collection.watch(full_document="updateLookup", max_await_time_ms=500) as stream:
        stream.try_next()
        start_token = stream.resume_token
    op_time = client.admin.command("ping").get("operationTime")
    snap_ts, snap_inc = (op_time.time, op_time.inc) if op_time else (int(time.time()), 0)

    writer = Writer(db, coll)
    for doc in collection.find({}, batch_size=1000):
        writer.add(doc_id_of(doc["_id"]), "snapshot", snap_ts, snap_inc, doc)
        if time.time() > deadline:
            raise TimeoutError("Snapshot exceeded RUN_SECONDS; raise RUN_SECONDS for the first run.")
    writer.flush()

    # Documents that existed before this snapshot and are absent from it were deleted while history was lost.
    session.sql(f"""
        UPDATE {SCHEMA}.DOCUMENTS d
        SET IS_DELETED = TRUE, UPDATED_AT = CURRENT_TIMESTAMP()
        WHERE d.SOURCE_DB = ? AND d.SOURCE_COLLECTION = ?
          AND d.IS_DELETED = FALSE
          AND NOT EXISTS (
              SELECT 1 FROM {SCHEMA}.RAW_EVENTS r
              WHERE r.RUN_ID = ? AND r.SOURCE_DB = d.SOURCE_DB
                AND r.SOURCE_COLLECTION = d.SOURCE_COLLECTION AND r.DOC_ID = d.DOC_ID)
    """, params=[db, coll, RUN_ID]).collect()

    save_checkpoint(db, coll, json_util.dumps(start_token), "OK", "snapshot complete",
                    snap_ts, snap_inc, caught_up=False, snapshot_done=True)
    return writer.total


def incremental(client, db, coll, token_json, deadline):
    collection = client[db][coll]
    writer = Writer(db, coll)
    last_ts = last_inc = None
    with collection.watch(full_document="updateLookup", resume_after=json_util.loads(token_json),
                          max_await_time_ms=1000, batch_size=1000) as stream:
        while time.time() < deadline:
            change = stream.try_next()
            if change is None:
                writer.flush()
                save_checkpoint(db, coll, json_util.dumps(stream.resume_token), "OK", None,
                                last_ts, last_inc, caught_up=True)
                return writer.total
            op = change["operationType"]
            if op in STOP_OPS:
                writer.flush()
                save_checkpoint(db, coll, json_util.dumps(stream.resume_token), "STOPPED",
                                f"stream ended by {op}; re-register the collection to restart",
                                last_ts, last_inc)
                return writer.total
            if op in CHANGE_OPS:
                ts = change["clusterTime"]
                last_ts, last_inc = ts.time, ts.inc
                writer.add(doc_id_of(change["documentKey"]["_id"]), op, ts.time, ts.inc,
                           change.get("fullDocument"))
            if len(writer.rows) == 0 and writer.seq % FLUSH_EVERY == 0 and writer.seq:
                save_checkpoint(db, coll, json_util.dumps(stream.resume_token), "OK", None, last_ts, last_inc)
        writer.flush()
        save_checkpoint(db, coll, json_util.dumps(stream.resume_token), "OK",
                        "time budget reached before catching up", last_ts, last_inc)
    return writer.total


collections = session.sql(f"""
    SELECT c.SOURCE_DB, c.SOURCE_COLLECTION, k.RESUME_TOKEN, k.STATUS
    FROM {SCHEMA}.COLLECTIONS c
    LEFT JOIN {SCHEMA}.CHECKPOINTS k
      ON k.SOURCE_DB = c.SOURCE_DB AND k.SOURCE_COLLECTION = c.SOURCE_COLLECTION
    WHERE c.ENABLED
    ORDER BY c.SOURCE_DB, c.SOURCE_COLLECTION
""").collect()

client = MongoClient(get_generic_secret_string(SECRET_PATH), serverSelectionTimeoutMS=15000)
deadline = time.time() + RUN_SECONDS
failures = []

for row in collections:
    db, coll, token, status = row["SOURCE_DB"], row["SOURCE_COLLECTION"], row["RESUME_TOKEN"], row["STATUS"]
    started = time.time()
    if status in ("HISTORY_LOST", "STOPPED"):
        log_run(db, coll, "skipped", 0, status, "needs attention, see CHECKPOINTS", started)
        failures.append(f"{db}.{coll}: {status}")
        continue
    mode = "incremental" if token else "snapshot"
    try:
        try:
            count = incremental(client, db, coll, token, deadline) if token else snapshot(client, db, coll, deadline)
        except OperationFailure as exc:
            if exc.code != HISTORY_LOST:
                raise
            if not AUTO_RESNAPSHOT:
                save_checkpoint(db, coll, token, "HISTORY_LOST",
                                "resume token is older than the oplog; set AUTO_RESNAPSHOT or re-snapshot")
                raise
            mode = "resnapshot"
            count = snapshot(client, db, coll, deadline)
        log_run(db, coll, mode, count, "OK", None, started)
        print(f"{db}.{coll}: {mode}, {count} rows")
    except (PyMongoError, TimeoutError) as exc:
        log_run(db, coll, mode, 0, "FAILED", str(exc)[:4000], started)
        failures.append(f"{db}.{coll}: {exc}")

client.close()
if failures:
    # Fails the scheduled run so it shows as failed in run history; earlier collections are already saved.
    raise RuntimeError("Sync failed for: " + "; ".join(failures))

## 5. Apply changes to current state

Consumes the stream on `RAW_EVENTS` and keeps one row per document. A row only replaces the current
one when its `(CLUSTER_TS, CLUSTER_INC)` is the same or newer, which is what makes replayed events
harmless. Updates whose `fullDocument` is null (the document was deleted before the lookup) are
skipped; the delete event that follows sets the final state.

In [ ]:
%%sql -r merge_documents
MERGE INTO SNOWFLAKE_EXAMPLE.MONGODB_CDC.DOCUMENTS t
USING (
    SELECT SOURCE_DB, SOURCE_COLLECTION, DOC_ID, OPERATION, DOCUMENT, CLUSTER_TS, CLUSTER_INC
    FROM SNOWFLAKE_EXAMPLE.MONGODB_CDC.RAW_EVENTS_STREAM
    WHERE OPERATION = 'delete' OR DOCUMENT IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY SOURCE_DB, SOURCE_COLLECTION, DOC_ID
        ORDER BY CLUSTER_TS DESC, CLUSTER_INC DESC,
                 IFF(OPERATION = 'snapshot', 0, 1) DESC, EVENT_SEQ DESC) = 1
) s
ON  t.SOURCE_DB = s.SOURCE_DB
AND t.SOURCE_COLLECTION = s.SOURCE_COLLECTION
AND t.DOC_ID = s.DOC_ID
WHEN MATCHED AND (s.CLUSTER_TS > t.CLUSTER_TS
                  OR (s.CLUSTER_TS = t.CLUSTER_TS AND s.CLUSTER_INC >= t.CLUSTER_INC)) THEN UPDATE SET
    DOCUMENT    = IFF(s.OPERATION = 'delete', t.DOCUMENT, s.DOCUMENT),
    IS_DELETED  = (s.OPERATION = 'delete'),
    CLUSTER_TS  = s.CLUSTER_TS,
    CLUSTER_INC = s.CLUSTER_INC,
    UPDATED_AT  = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
    (SOURCE_DB, SOURCE_COLLECTION, DOC_ID, DOCUMENT, IS_DELETED, CLUSTER_TS, CLUSTER_INC, UPDATED_AT)
VALUES
    (s.SOURCE_DB, s.SOURCE_COLLECTION, s.DOC_ID, s.DOCUMENT, s.OPERATION = 'delete',
     s.CLUSTER_TS, s.CLUSTER_INC, CURRENT_TIMESTAMP());

## 6. Typed column views from `COLUMN_MAP`

One view per registered collection, generated from the mapping rows, with deleted documents
filtered out. Add a row to `COLUMN_MAP` and the next run adds the column. Names, paths and types are
checked against allowlists before they reach SQL.

In [ ]:
import re

ALLOWED_TYPES = {"STRING", "NUMBER", "FLOAT", "BOOLEAN", "DATE", "TIMESTAMP_TZ", "TIMESTAMP_NTZ", "VARIANT", "ARRAY", "OBJECT"}
IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{0,254}$")
PATH = re.compile(r"^[A-Za-z0-9_$.\[\]]{1,500}$")

mappings = session.sql(f"""
    SELECT c.SOURCE_DB, c.SOURCE_COLLECTION, c.TARGET_VIEW, m.COLUMN_NAME, m.JSON_PATH, m.DATA_TYPE
    FROM {SCHEMA}.COLLECTIONS c
    JOIN {SCHEMA}.COLUMN_MAP m
      ON m.SOURCE_DB = c.SOURCE_DB AND m.SOURCE_COLLECTION = c.SOURCE_COLLECTION
    WHERE c.ENABLED
    ORDER BY c.TARGET_VIEW, m.ORDINAL, m.COLUMN_NAME
""").collect()

views = {}
for m in mappings:
    views.setdefault((m["SOURCE_DB"], m["SOURCE_COLLECTION"], m["TARGET_VIEW"]), []).append(m)

for (db, coll, view), cols in views.items():
    problems = [c["COLUMN_NAME"] for c in cols
                if not IDENT.match(c["COLUMN_NAME"]) or not PATH.match(c["JSON_PATH"])
                or c["DATA_TYPE"].upper() not in ALLOWED_TYPES]
    if not IDENT.match(view) or problems:
        print(f"Skipped {view}: invalid view name or mapping rows {problems}. Fix COLUMN_MAP and rerun.")
        continue
    # Date-typed paths read the relaxed Extended JSON form {"$date": "..."} as well as plain values.
    select_cols = ",\n    ".join(
        f"TRY_CAST(COALESCE(GET_PATH(DOCUMENT, '{c['JSON_PATH']}.\"$date\"'), GET_PATH(DOCUMENT, '{c['JSON_PATH']}'))::STRING AS {c['DATA_TYPE'].upper()}) AS {c['COLUMN_NAME']}"
        if c["DATA_TYPE"].upper() in ("DATE", "TIMESTAMP_TZ", "TIMESTAMP_NTZ")
        else f"GET_PATH(DOCUMENT, '{c['JSON_PATH']}')::{c['DATA_TYPE'].upper()} AS {c['COLUMN_NAME']}"
        for c in cols)
    session.sql(f"""
        CREATE OR REPLACE VIEW {SCHEMA}.{view} AS
        SELECT
            DOC_ID,
            {select_cols},
            DOCUMENT,
            TO_TIMESTAMP_TZ(CLUSTER_TS) AS SOURCE_CHANGED_AT,
            UPDATED_AT
        FROM {SCHEMA}.DOCUMENTS
        WHERE SOURCE_DB = ? AND SOURCE_COLLECTION = ? AND NOT IS_DELETED
    """, params=[db, coll]).collect()
    print(f"View {SCHEMA}.{view}: {len(cols)} mapped columns")

## 7. Lag and health

`CAUGHT_UP_AT` is the last moment a run reached the head of the change stream, so data is at most
`SECONDS_SINCE_CAUGHT_UP` behind. `RESUME_TOKEN` is the stream position itself.

In [ ]:
%%sql -r create_lag_view
CREATE OR REPLACE VIEW SNOWFLAKE_EXAMPLE.MONGODB_CDC.SYNC_LAG AS
SELECT
    c.SOURCE_DB,
    c.SOURCE_COLLECTION,
    k.STATUS,
    k.STATUS_MESSAGE,
    k.CAUGHT_UP_AT,
    DATEDIFF('second', k.CAUGHT_UP_AT, CURRENT_TIMESTAMP()) AS SECONDS_SINCE_CAUGHT_UP,
    TO_TIMESTAMP_TZ(k.LAST_EVENT_CLUSTER_TS) AS LAST_EVENT_SOURCE_TIME,
    k.SNAPSHOT_COMPLETED_AT,
    k.UPDATED_AT AS LAST_CHECKPOINT_AT,
    k.RESUME_TOKEN
FROM SNOWFLAKE_EXAMPLE.MONGODB_CDC.COLLECTIONS c
LEFT JOIN SNOWFLAKE_EXAMPLE.MONGODB_CDC.CHECKPOINTS k
  ON k.SOURCE_DB = c.SOURCE_DB AND k.SOURCE_COLLECTION = c.SOURCE_COLLECTION
WHERE c.ENABLED;

In [ ]:
%%sql -r sync_status
SELECT SOURCE_DB, SOURCE_COLLECTION, STATUS, SECONDS_SINCE_CAUGHT_UP, LAST_EVENT_SOURCE_TIME, LAST_CHECKPOINT_AT
FROM SNOWFLAKE_EXAMPLE.MONGODB_CDC.SYNC_LAG
ORDER BY SECONDS_SINCE_CAUGHT_UP DESC NULLS FIRST;

## 8. Schedule it

1. Run the notebook once interactively. The first run snapshots each collection; for a large
   collection raise `RUN_SECONDS` for that run.
2. Select the calendar icon, choose a frequency (for example every 5 minutes), and attach
   `MONGODB_CDC_EAI` and the `MONGODB_URI` secret in the schedule dialog.
3. After editing the notebook, select **Deploy Changes** so scheduled runs pick up the new version.

Scheduled runs use your default role, and failures appear in the notebook's run history because
section 4 raises when any collection fails.

## Teardown

```sql
DROP SCHEMA IF EXISTS SNOWFLAKE_EXAMPLE.MONGODB_CDC;   -- tables, stream, views, secret, rules
DROP INTEGRATION IF EXISTS MONGODB_CDC_EAI;            -- needs the integration owner
```